# Figure 9 — Rate histograms in Erdős–Rényi networks

Pena, Vellmer, Bernardi, Roque & Lindner (2018), Front. Comput. Neurosci. 12:9, doi:10.3389/fncom.2018.00009. Scheme after Dummer, Wieland & Lindner (2014).

**Caption (paper):** Firing-rate histograms of the scheme with M = 50 neurons per generation (symbols) and the network (dashed) for (A) binomially distributed in-degrees and (B) additionally exponential weights. Scheme histograms use generations 6–20. Parameters as in Figure 10.

**Method here:** ER scheme of Section 2.5 (Eqs. 16–18; Eq. 18 shift for g = 6) and an ER network (p = C/N) with N_E = 10,000. All g of Figure 10 are shown.

**Shared implementation choices** (all figures, see `pena_scheme.py`, `pena_network.py`):
- Euler, `dt = 0.1 ms`; voltage clamped at reset during the refractory period (scheme and network).
- Spectra: Eq. 5, two-sided, 2 s windows after a 1 s transient (Δf = 0.5 Hz).
  Fano factor = mean of S at 0.5 and 1 Hz divided by ν; τc from Eq. 7 with the estimator variance subtracted.
- Scheme generation 0: white noise at the diffusion-approximation (mean-field) rate; the paper does not
  specify its starting point. Inhibition-dominated cases (g > 4) use the all-past rate average (Eq. 11).
- Networks are simulated at reduced size (default N_E = 10,000 instead of 100,000) to fit 2 CPUs / ~2 GB;
  partners are drawn with replacement. Larger sizes are available via the config cell but need far more memory.

In [1]:
from pathlib import Path
import sys, json, time
import numpy as np
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "pena2018_params.py").exists():
    HERE = HERE / "05_Pena2018_Replication"
sys.path.insert(0, str(HERE))

from pena2018_params import params_for
import pena_scheme as ps
import pena_runs as pr
FIGURES = HERE / "figures"; FIGURES.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [2]:
G_VALUES = [3.5, 4.0, 6.0]
M_VALUES = [10, 50, 100]
N_E_NET, N_WINDOWS = 10_000, 10        # paper: N_E = 100,000
SCHEME = dict(n_generations=20, n_trials=40)   # paper: 100 trials of 10 s per neuron

In [ ]:
sch, net = {}, {}
for weights in ['fixed', 'exponential']:
    for g in G_VALUES:
        print(f"--- {weights} weights, g = {g}")
        net[weights, g] = pr.er_network(g, weights, N_E_NET, N_WINDOWS)
        for M in M_VALUES:
            sch[weights, g, M] = pr.er_scheme(g, weights, M, **SCHEME)

--- fixed weights, g = 3.5
loaded net_er_a2ba1a03fe.npz
loaded scheme_er_ec5a38ae14.npz
loaded scheme_er_ecc3184410.npz
loaded scheme_er_4b1e0b6ef8.npz
--- fixed weights, g = 4.0
loaded net_er_8c4f3321ae.npz
loaded scheme_er_922b73a91c.npz
loaded scheme_er_9deb55beb8.npz
gen  0: mean rate   63.65 Hz, SD  29.74 Hz
gen  1: mean rate   65.08 Hz, SD  33.59 Hz
gen  2: mean rate   75.54 Hz, SD  31.33 Hz
gen  3: mean rate   64.48 Hz, SD  36.67 Hz
gen  4: mean rate   65.20 Hz, SD  34.74 Hz
gen  5: mean rate   67.99 Hz, SD  32.46 Hz
gen  6: mean rate   68.80 Hz, SD  35.65 Hz
gen  7: mean rate   64.23 Hz, SD  36.87 Hz


In [1]:
fig, axes = plt.subplots(2, len(G_VALUES), figsize=(13, 6.5))
for r, weights in enumerate(["fixed", "exponential"]):
    for c, g in enumerate(G_VALUES):
        ax = axes[r, c]
        n = net[weights, g]; rates_net = n["rates"][: int(n["offsets"][1])]
        s = sch[weights, g, 50]; rates_sch = s["rates"][6:21].ravel()
        bins = np.linspace(0, np.percentile(rates_net, 99.5), 30)
        hn, _ = np.histogram(rates_net, bins, density=True)
        hs, _ = np.histogram(rates_sch, bins, density=True)
        mid = 0.5 * (bins[1:] + bins[:-1])
        ax.plot(mid, hn, "--", color="k", label="network")
        ax.plot(mid, hs, "o", ms=4, color="tab:blue", label="scheme (M=50, gen 6-20)")
        ax.set(title=f"{'AB'[r]}  {weights} weights, g = {g}", xlabel="rate (Hz)", ylabel="density")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIGURES / "figure09_rate_histograms.png", dpi=200); plt.show()
for weights in ["fixed", "exponential"]:
    for g in G_VALUES:
        n = net[weights, g]; rn = n["rates"][: int(n["offsets"][1])]; rs = sch[weights, g, 50]["rates"][6:21].ravel()
        print(f"{weights:11s} g={g}: mean {rs.mean():6.2f}/{rn.mean():6.2f} Hz, SD {rs.std():6.2f}/{rn.std():6.2f} Hz (scheme/network)")

NameError: name 'plt' is not defined

## Notes

_Filled in after the run._